In [1]:
import os, csv, random
from collections import defaultdict, Counter

KG_DIR     = r'D:\Memoire\LLMKGER\kg'
OUTPUT_DIR = r'D:\Memoire\LLMKGER\kg_sampled'
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(42)
print('Prets')


Prets


In [2]:
# Charger tous les triplets en memoire par relation
from collections import defaultdict

triples_by_rel = defaultdict(list)

with open(os.path.join(KG_DIR, 'triplets.tsv'), encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        triples_by_rel[row['relation']].append((row['head'], row['relation'], row['tail']))

print('Triplets charges par relation :')
for rel, triples in triples_by_rel.items():
    print(f'  {rel:<20} {len(triples):>10,}')


Triplets charges par relation :
  watches               1,347,769
  has_concept             624,683
  belongs_to              451,078
  taught_by                13,312
  offered_by                3,605
  follows              11,523,119
  prerequisite_of          84,968


In [3]:
MIN_INTERACTIONS = 5   # user doit avoir suivi >= 5 cours
MIN_USERS        = 10  # cours doit avoir >= 10 users

# Compter interactions par user et par cours
user_course_count = Counter()
course_user_count = Counter()

for h, r, t in triples_by_rel['follows']:
    user_course_count[h] += 1
    course_user_count[t] += 1

# Filtrer
active_users  = {u for u, cnt in user_course_count.items() if cnt >= MIN_INTERACTIONS}
active_courses = {c for c, cnt in course_user_count.items() if cnt >= MIN_USERS}

print(f'Seuils : MIN_INTERACTIONS={MIN_INTERACTIONS}, MIN_USERS={MIN_USERS}')
print(f'Users  avant : {len(user_course_count):,}  ->  apres : {len(active_users):,}')
print(f'Cours  avant : {len(course_user_count):,}  ->  apres : {len(active_courses):,}')


Seuils : MIN_INTERACTIONS=5, MIN_USERS=10
Users  avant : 3,211,048  ->  apres : 373,369
Cours  avant : 3,781  ->  apres : 3,055


In [4]:
# Filtrer les triplets follows
follows_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['follows']
    if h in active_users and t in active_courses
]
print(f'follows : {len(triples_by_rel["follows"]):,}  ->  {len(follows_filtered):,}')

# Propager : garder seulement watches des users actifs
watches_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['watches']
    if h in active_users
]
print(f'watches : {len(triples_by_rel["watches"]):,}  ->  {len(watches_filtered):,}')

# CCIDs gardes (videos vues par les users actifs)
active_ccids = {t for h, r, t in watches_filtered}
print(f'CCIDs actifs : {len(active_ccids):,}')

# has_concept : garder seulement CCIDs actifs
has_concept_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['has_concept']
    if h in active_ccids
]
print(f'has_concept : {len(triples_by_rel["has_concept"]):,}  ->  {len(has_concept_filtered):,}')

# Concepts actifs
active_concepts = {t for h, r, t in has_concept_filtered}
print(f'Concepts actifs : {len(active_concepts):,}')

# belongs_to : garder concepts actifs ET cours actifs
belongs_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['belongs_to']
    if h in active_concepts and t in active_courses
]
print(f'belongs_to : {len(triples_by_rel["belongs_to"]):,}  ->  {len(belongs_filtered):,}')

# prerequisite_of : garder concepts actifs
prereq_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['prerequisite_of']
    if h in active_concepts and t in active_concepts
]
print(f'prerequisite_of : {len(triples_by_rel["prerequisite_of"]):,}  ->  {len(prereq_filtered):,}')

# taught_by et offered_by : garder cours actifs
taught_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['taught_by']
    if h in active_courses
]
offered_filtered = [
    (h, r, t) for h, r, t in triples_by_rel['offered_by']
    if h in active_courses
]
print(f'taught_by   : {len(triples_by_rel["taught_by"]):,}  ->  {len(taught_filtered):,}')
print(f'offered_by  : {len(triples_by_rel["offered_by"]):,}  ->  {len(offered_filtered):,}')


follows : 11,523,119  ->  7,470,566
watches : 1,347,769  ->  436,710
CCIDs actifs : 28,439
has_concept : 624,683  ->  389,479
Concepts actifs : 143,751
belongs_to : 451,078  ->  278,385
prerequisite_of : 84,968  ->  72,361
taught_by   : 13,312  ->  11,255
offered_by  : 3,605  ->  3,010


In [5]:
sampled_relations = {
    'watches'        : watches_filtered,
    'has_concept'    : has_concept_filtered,
    'belongs_to'     : belongs_filtered,
    'taught_by'      : taught_filtered,
    'offered_by'     : offered_filtered,
    'follows'        : follows_filtered,
    'prerequisite_of': prereq_filtered,
}
total = sum(len(v) for v in sampled_relations.values())

print('=' * 50)
print('  KG Sampled — Statistiques')
print('=' * 50)
print('\nNOEUDS :')
print(f'  Users    {len(active_users):>10,}')
print(f'  Courses  {len(active_courses):>10,}')
print(f'  Videos   {len(active_ccids):>10,}')
print(f'  Concepts {len(active_concepts):>10,}')
print('\nARETES :')
for rel, triples in sampled_relations.items():
    print(f'  {rel:<20} {len(triples):>10,}')
print(f'  {"TOTAL":<20} {total:>10,}')
print('=' * 50)
print('\n=> Si TOTAL > 2M, augmenter MIN_INTERACTIONS ou MIN_USERS')
print('=> Si TOTAL < 200K, diminuer les seuils')


  KG Sampled — Statistiques

NOEUDS :
  Users       373,369
  Courses       3,055
  Videos       28,439
  Concepts    143,751

ARETES :
  watches                 436,710
  has_concept             389,479
  belongs_to              278,385
  taught_by                11,255
  offered_by                3,010
  follows               7,470,566
  prerequisite_of          72,361
  TOTAL                 8,661,766

=> Si TOTAL > 2M, augmenter MIN_INTERACTIONS ou MIN_USERS
=> Si TOTAL < 200K, diminuer les seuils


## Étape 2 — Split train / valid / test


In [6]:
# Split Leave-One-Out sur les interactions follows
# Pour chaque user : 1 interaction en test, 1 en valid, reste en train

user_interactions = defaultdict(list)
for h, r, t in follows_filtered:
    user_interactions[h].append(t)

train_interactions = []
valid_interactions = []
test_interactions  = []

for uid, courses_list in user_interactions.items():
    random.shuffle(courses_list)
    if len(courses_list) >= 3:
        test_interactions.append( (uid, 'follows', courses_list[0]))
        valid_interactions.append((uid, 'follows', courses_list[1]))
        for c in courses_list[2:]:
            train_interactions.append((uid, 'follows', c))
    else:
        # Trop peu d'interactions -> tout en train
        for c in courses_list:
            train_interactions.append((uid, 'follows', c))

print(f'Split LOO (Leave-One-Out) :')
print(f'  Train : {len(train_interactions):,}')
print(f'  Valid : {len(valid_interactions):,}')
print(f'  Test  : {len(test_interactions):,}')


Split LOO (Leave-One-Out) :
  Train : 6,723,828
  Valid : 373,369
  Test  : 373,369


In [7]:
# ── Export kg_sampled/triplets_all.tsv (KG complet sample, sans follows) ──
# Les follows sont dans les splits train/valid/test
kg_triples = []
for rel, triples in sampled_relations.items():
    if rel != 'follows':
        kg_triples.extend(triples)
# Ajouter les follows de train seulement dans le KG
kg_triples.extend(train_interactions)

with open(os.path.join(OUTPUT_DIR, 'triplets_train_kg.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['head', 'relation', 'tail'])
    for h, r, t in kg_triples:
        w.writerow([h, r, t])
print(f'triplets_train_kg.tsv : {len(kg_triples):,} triplets')

# ── Export splits ─────────────────────────────────────────────────────────
for fname, data in [('train.tsv', train_interactions),
                     ('valid.tsv', valid_interactions),
                     ('test.tsv',  test_interactions)]:
    with open(os.path.join(OUTPUT_DIR, fname), 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(['user_id', 'relation', 'course_id'])
        for h, r, t in data:
            w.writerow([h, r, t])
    print(f'{fname} : {len(data):,} interactions')

# ── Export entity2id et relation2id (pour les modeles KGE) ───────────────
all_entities = set()
for h, r, t in kg_triples:
    all_entities.add(h)
    all_entities.add(t)

entity2id = {e: i for i, e in enumerate(sorted(all_entities))}
relation2id = {r: i for i, r in enumerate(sorted(sampled_relations.keys()))}

with open(os.path.join(OUTPUT_DIR, 'entity2id.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['entity', 'id'])
    for e, i in entity2id.items():
        w.writerow([e, i])

with open(os.path.join(OUTPUT_DIR, 'relation2id.tsv'), 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['relation', 'id'])
    for r, i in relation2id.items():
        w.writerow([r, i])

print(f'\nentity2id  : {len(entity2id):,} entites')
print(f'relation2id: {len(relation2id):,} relations')

print('\nFichiers dans', OUTPUT_DIR)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = round(os.path.getsize(fpath)/1024/1024, 1)
    print(f'  {fname:<42} {size} MB')


triplets_train_kg.tsv : 7,915,028 triplets
train.tsv : 6,723,828 interactions
valid.tsv : 373,369 interactions
test.tsv : 373,369 interactions

entity2id  : 558,364 entites
relation2id: 7 relations

Fichiers dans D:\Memoire\LLMKGER\kg_sampled
  entity2id.tsv                              13.7 MB
  relation2id.tsv                            0.0 MB
  test.tsv                                   10.3 MB
  train.tsv                                  186.1 MB
  triplets_train_kg.tsv                      257.6 MB
  valid.tsv                                  10.3 MB
